# Session 2 Hands-on Lab Notebook
## AWS data engineering for AI systems: scoping, collection, preprocessing, labeling, storage, ingestion, and versioning

 
**Runs on:** A real AWS account from SageMaker Studio, SageMaker Notebook Instance, or a Jupyter environment with AWS credentials.  
**No simulation:** The notebook calls real AWS services through `boto3`, the AWS SDK, and the SageMaker SDK.

This notebook implements what Session 1 taught:

1. AI problem scoping and optional Bedrock-based role conversation.
2. Data collection into Amazon S3 with versioning and immutable manifests.
3. Data profiling, cleaning, validation, and AWS Glue Data Quality.
4. AWS Glue Data Catalog, crawlers, and Amazon Athena SQL profiling.
5. AWS Glue ETL job that writes curated Parquet data.
6. Programmatic labels and an optional SageMaker Ground Truth labeling job.
7. Optional Bedrock embeddings for retrieval-ready metadata.
8. Cleanup so the lab does not leave unexpected cost.

The case study uses the public UCI Bank Marketing dataset as a human-centric AI decision example. The goal is to predict whether a customer subscribes to a term deposit. This dataset is useful because it demonstrates tabular data engineering, class imbalance, labeling, versioning, and later bias/explainability work.


## 0. Lab safety, prerequisites, and expected AWS services

### Services used

Core services used in this notebook:

- Amazon S3
- AWS IAM
- AWS Glue Data Catalog
- AWS Glue crawlers
- AWS Glue ETL jobs
- AWS Glue Data Quality
- Amazon Athena
- Amazon SageMaker Ground Truth (optional but included)
- Amazon Bedrock runtime (optional but included)
- Amazon CloudWatch Logs (created by Glue jobs)

### Permissions

You need an AWS role or user with permissions to use S3, Glue, Athena, SageMaker, and optional IAM PassRole. In a training environment, the instructor should pre-create the following and paste them in the configuration cell:

- `LAB_BUCKET` or permission to create a bucket.
- `GLUE_ROLE_ARN` trusted by `glue.amazonaws.com` with S3 read/write permissions for this lab bucket.
- `SAGEMAKER_EXECUTION_ROLE_ARN` for SageMaker and Ground Truth.
- Optional `GROUND_TRUTH_WORKTEAM_ARN` for a private workforce.
- Optional Bedrock model access in the selected region.

### Cost controls

Glue jobs, crawlers, Athena queries, Ground Truth, and Bedrock can incur cost. The notebook includes a cleanup section. Do not skip cleanup in a shared training account.


In [0]:
import subprocess, sys

pins = [
    "sagemaker>=2.200.0,<3.0.0",   # 3.x removed sagemaker.Session — stay on 2.x
    "numpy>=1.25.0,<2.4.0",        # 2.4.6 breaks autogluon, sagemaker-studio, catboost
    "protobuf>=3.20.3,<6.0.0",     # 6.x breaks tensorflow + grpcio-status
    "pyarrow>=7.0.0,<21.0.0",      # 24.x breaks autogluon-common
    "scikit-learn>=1.4.0,<1.8.0",  # 1.9.x breaks autogluon-tabular/core/features
    "aiobotocore[boto3]",           # re-resolve botocore compat after boto3 upgrade
]

print("Pinning conflicting packages...")
for pkg in pins:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        capture_output=True, text=True
    )
    print(f"  {'✅' if r.returncode == 0 else '❌'} {pkg}")

print("\n✅ Done. Kernel → Restart Kernel → Run All.")

Pinning conflicting packages...


  ✅ sagemaker>=2.200.0,<3.0.0


  ✅ numpy>=1.25.0,<2.4.0


  ✅ protobuf>=3.20.3,<6.0.0


  ✅ pyarrow>=7.0.0,<21.0.0


  ✅ scikit-learn>=1.4.0,<1.8.0


  ✅ aiobotocore[boto3]

✅ Done. Kernel → Restart Kernel → Run All.


In [0]:
import os
import io
import re
import json
import time
import uuid
import zipfile
import hashlib
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import boto3
import botocore
import pandas as pd
import numpy as np
import awswrangler as wr

try:
    import sagemaker
except Exception:
    sagemaker = None

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)


sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


## 1. Configure the lab

Set `I_UNDERSTAND_COSTS = True` only when the instructor has confirmed that the AWS account and permissions are ready.

The default configuration uses one unique lab id so that resources from different learners do not collide.


In [0]:
I_UNDERSTAND_COSTS = True  

REGION = os.environ.get('AWS_REGION') or os.environ.get('AWS_DEFAULT_REGION') or boto3.Session().region_name or 'us-east-1'
ACCOUNT_ID = boto3.client('sts', region_name=REGION).get_caller_identity()['Account']
CALLER_ARN = boto3.client('sts', region_name=REGION).get_caller_identity()['Arn']

LAB_ID = os.environ.get('LAB_ID', f"bits-ai-de-{ACCOUNT_ID[-4:]}-{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}")

# Use an existing bucket if your lab account has a policy against creating buckets.
LAB_BUCKET = os.environ.get('LAB_BUCKET', f"{LAB_ID}-{REGION}").lower().replace('_','-')
PREFIX = os.environ.get('LAB_PREFIX', 'module2/session2')

# Service roles. Instructors should pre-create these in locked-down accounts.
GLUE_ROLE_ARN = os.environ.get('GLUE_ROLE_ARN', '')
SAGEMAKER_EXECUTION_ROLE_ARN = os.environ.get('SAGEMAKER_EXECUTION_ROLE_ARN', '')
GROUND_TRUTH_WORKTEAM_ARN = os.environ.get('GROUND_TRUTH_WORKTEAM_ARN', '')

# Optional features. Keep costly or account-specific features off until configured.
RUN_BEDROCK_SCOPING = False
RUN_BEDROCK_EMBEDDINGS = False
RUN_GROUND_TRUTH_JOB = False
RUN_GLUE_DATA_QUALITY = True
RUN_GLUE_ETL_JOB = True

print('Region:', REGION)
print('Account:', ACCOUNT_ID)
print('Caller:', CALLER_ARN)
print('Lab id:', LAB_ID)
print('Bucket:', LAB_BUCKET)
print('Prefix:', PREFIX)

if not I_UNDERSTAND_COSTS:
    print("\n⚠️  REMINDER: Set I_UNDERSTAND_COSTS = True after the instructor confirms account, role, and cost readiness. The next cell will assert on this.")


Region: us-east-1
Account: 483955931464
Caller: arn:aws:sts::483955931464:assumed-role/AmazonSageMakerAdminIAMExecutionRole/25d1cf33-5d78-4ec1-abf2-a12d72a08515
Lab id: bits-ai-de-1464-20260721161520
Bucket: bits-ai-de-1464-20260721161520-us-east-1
Prefix: module2/session2


In [0]:
assert I_UNDERSTAND_COSTS, "Set I_UNDERSTAND_COSTS=True after the instructor confirms account, role, and cost readiness."


## 2. Create AWS clients and helper functions

These helpers make service waits visible and write evidence files to S3. The lab intentionally stores artifacts, not just notebook outputs.


In [0]:
session = boto3.Session(region_name=REGION)
s3 = session.client('s3')
glue = session.client('glue')
athena = session.client('athena')
sts = session.client('sts')
sagemaker_client = session.client('sagemaker')

try:
    sm_session = sagemaker.Session(boto_session=session)
    print("✅ SageMaker session created:", sm_session.boto_region_name)
except AttributeError:
    import importlib
    importlib.reload(sagemaker)
    try:
        sm_session = sagemaker.Session(boto_session=session)
        print("✅ SageMaker session created after reload:", sm_session.boto_region_name)
    except Exception as e:
        sm_session = None
        print(f"⚠️  SageMaker session unavailable after reload: {e}")
        print("   Attempting direct re-import...")
        try:
            import importlib, sys
            # Remove stale cached module entirely and re-import fresh
            for mod in [k for k in sys.modules if k.startswith('sagemaker')]:
                sys.modules.pop(mod)
            import sagemaker
            sm_session = sagemaker.Session(boto_session=session)
            print("✅ SageMaker session created after full re-import:", sm_session.boto_region_name)
        except Exception as e2:
            sm_session = None
            print(f"⚠️  SageMaker unavailable: {e2}")
            print("   Non-SageMaker cells will still work normally.")
            print("   Fix: Kernel → Restart Kernel → Run All Cells.")

LOCAL_DIR = Path('/tmp/bits_ai_de_session2')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

def s3_uri(*parts):
    clean = [str(p).strip('/').replace(' ', '_') for p in parts if str(p).strip('/')]
    return 's3://' + LAB_BUCKET + '/' + '/'.join(clean)

def put_json_s3(obj, key):
    s3.put_object(
        Bucket=LAB_BUCKET,
        Key=key,
        Body=json.dumps(obj, indent=2, default=str).encode('utf-8'),
        ContentType='application/json'
    )
    return f's3://{LAB_BUCKET}/{key}'

def put_text_s3(text, key, content_type='text/plain'):
    s3.put_object(Bucket=LAB_BUCKET, Key=key, Body=text.encode('utf-8'), ContentType=content_type)
    return f's3://{LAB_BUCKET}/{key}'

def md5_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def wait_for_crawler(name):
    while True:
        state = glue.get_crawler(Name=name)['Crawler']['State']
        print('Crawler state:', state)
        if state == 'READY':
            return
        time.sleep(15)

def wait_for_glue_job(job_name, run_id):
    while True:
        run = glue.get_job_run(JobName=job_name, RunId=run_id)['JobRun']
        state = run['JobRunState']
        print('Glue job state:', state)
        if state in ['SUCCEEDED', 'FAILED', 'STOPPED', 'TIMEOUT', 'ERROR']:
            return run
        time.sleep(30)

def wait_for_athena(query_execution_id):
    while True:
        response = athena.get_query_execution(QueryExecutionId=query_execution_id)
        state = response['QueryExecution']['Status']['State']
        print('Athena state:', state)
        if state in ['SUCCEEDED', 'FAILED', 'CANCELLED']:
            return response
        time.sleep(2)

def start_athena_query(sql, database=None):
    result_location = s3_uri(PREFIX, 'athena-results') + '/'
    params = {
        'QueryString': sql,
        'ResultConfiguration': {'OutputLocation': result_location},
    }
    if database:
        params['QueryExecutionContext'] = {'Database': database}
    qid = athena.start_query_execution(**params)['QueryExecutionId']
    result = wait_for_athena(qid)
    if result['QueryExecution']['Status']['State'] != 'SUCCEEDED':
        raise RuntimeError(result['QueryExecution']['Status'])
    return qid

def athena_to_df(sql, database=None):
    # Use awswrangler directly to avoid running the query twice
    return wr.athena.read_sql_query(
        sql,
        database=database,
        boto3_session=session,
        s3_output=s3_uri(PREFIX, 'athena-results') + '/',
        ctas_approach=False,
    )

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


✅ SageMaker session created: us-east-1


## 3. Create the lab S3 bucket with versioning and encryption

This implements the Session 1 storage and versioning theory:

- S3 bucket as data lake root.
- Versioning for reproducibility.
- Server-side encryption.
- Public access block.
- Structured prefixes for raw, validated, curated, labels, audit, scripts, and Athena results.


In [0]:
def bucket_exists(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return True
    except botocore.exceptions.ClientError:
        return False

if not bucket_exists(LAB_BUCKET):
    print('Creating bucket', LAB_BUCKET)
    if REGION == 'us-east-1':
        s3.create_bucket(Bucket=LAB_BUCKET)
    else:
        s3.create_bucket(Bucket=LAB_BUCKET, CreateBucketConfiguration={'LocationConstraint': REGION})
else:
    print('Using existing bucket', LAB_BUCKET)

s3.put_public_access_block(
    Bucket=LAB_BUCKET,
    PublicAccessBlockConfiguration={
        'BlockPublicAcls': True,
        'IgnorePublicAcls': True,
        'BlockPublicPolicy': True,
        'RestrictPublicBuckets': True,
    },
)

s3.put_bucket_versioning(Bucket=LAB_BUCKET, VersioningConfiguration={'Status': 'Enabled'})
s3.put_bucket_encryption(
    Bucket=LAB_BUCKET,
    ServerSideEncryptionConfiguration={
        'Rules': [{'ApplyServerSideEncryptionByDefault': {'SSEAlgorithm': 'AES256'}}]
    },
)

try:
    s3.put_bucket_tagging(
        Bucket=LAB_BUCKET,
        Tagging={'TagSet': [
            {'Key': 'project', 'Value': 'bits-ai-mlops-module2'},
            {'Key': 'session', 'Value': 'session2'},
            {'Key': 'owner', 'Value': 'training'},
            {'Key': 'delete-after', 'Value': 'lab-cleanup'},
        ]}
    )
except Exception as e:
    print('Tagging skipped:', e)

for p in ['raw', 'validated', 'curated', 'labels', 'audit', 'scripts', 'athena-results', 'glue-dq', 'bedrock', 'ground-truth']:
    s3.put_object(Bucket=LAB_BUCKET, Key=f'{PREFIX}/{p}/.keep', Body=b'')

print('S3 data lake root:', s3_uri(PREFIX))


Creating bucket bits-ai-de-1464-20260721161520-us-east-1


S3 data lake root: s3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2


S3 data lake root: s3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2


## 4. Collect a real public dataset

The source is the UCI Bank Marketing dataset. We download it, preserve the raw file, and add a small amount of operational metadata for the lab. In a production project, this step would be replaced by a controlled ingestion from a source system, API, partner file drop, or CDC stream.


In [0]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
zip_path = LOCAL_DIR / 'bank_marketing.zip'
extract_dir = LOCAL_DIR / 'uci_bank_marketing'
extract_dir.mkdir(exist_ok=True)

raw_csv = None

# Tier 1: Direct download from UCI
try:
    print('Tier 1: Downloading UCI dataset from the internet...')
    urllib.request.urlretrieve(DATA_URL, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_dir)
        print('Outer zip contents:', z.namelist())

    # The outer zip contains bank-additional.zip (semicolon-separated, 21 features)
    # and bank.zip (comma-separated, 17 features). We always want bank-additional-full.csv.
    nested_zip = extract_dir / 'bank-additional.zip'
    if not nested_zip.exists():
        # Some versions nest it one level deeper
        nested_zip = next(extract_dir.rglob('bank-additional.zip'), None)

    if nested_zip and nested_zip.exists():
        inner_dir = extract_dir / 'bank-additional'
        inner_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(nested_zip, 'r') as z:
            z.extractall(inner_dir)
            print('Inner zip contents:', z.namelist())
        candidate = inner_dir / 'bank-additional-full.csv'
        if not candidate.exists():
            # Some versions extract into a subdirectory
            candidate = next(inner_dir.rglob('bank-additional-full.csv'), None)
    else:
        # Fallback: search entire extract tree
        candidate = next(extract_dir.rglob('bank-additional-full.csv'), None)

    if candidate and candidate.exists():
        raw_csv = candidate
        print('Tier 1 succeeded:', raw_csv)
    else:
        raise FileNotFoundError(f'bank-additional-full.csv not found under {extract_dir}')

except Exception as e:
    print(f'Tier 1 failed (likely VPC/network restriction): {e}')

# Tier 2: SageMaker sample-files bucket (us-east-1 only)
if raw_csv is None:
    try:
        print('Tier 2: Trying SageMaker sample-files S3 bucket...')
        _sample_csv = LOCAL_DIR / 'bank-additional-full.csv'
        s3.download_file('sagemaker-sample-files',
                         'datasets/tabular/uci_bank_marketing/bank-additional-full.csv',
                         str(_sample_csv))
        if _sample_csv.exists():
            raw_csv = _sample_csv
            print('Tier 2 succeeded:', raw_csv)
    except Exception as e:
        print(f'Tier 2 failed (bucket may not exist in region {REGION}): {e}')

# Tier 3: Manual upload fallback
if raw_csv is None:
    _manual = LOCAL_DIR / 'bank-additional-full.csv'
    if _manual.exists():
        raw_csv = _manual
        print(f'Tier 3: Found manually uploaded file at {raw_csv}')
    else:
        raise FileNotFoundError(
            "\n\n❌ All download attempts failed.\n"
            "Please manually upload 'bank-additional-full.csv' to this path and re-run:\n"
            f"  {str(_manual)}\n"
            "Download the file from: https://archive.ics.uci.edu/dataset/222/bank+marketing\n"
            "  → extract bank+marketing.zip → extract bank-additional.zip → use bank-additional-full.csv"
        )

assert raw_csv.exists(), 'Dataset file not found after all fallbacks.'
raw_md5 = md5_file(raw_csv)
print('Raw CSV:', raw_csv)
print('Raw MD5:', raw_md5)

# Auto-detect separator — handles both ';' and ',' variants defensively
import csv
with open(raw_csv, 'r') as f:
    dialect = csv.Sniffer().sniff(f.read(2048), delimiters=';,')
sep = dialect.delimiter
print(f'Detected separator: "{sep}"')

df = pd.read_csv(raw_csv, sep=sep)
print('Shape:', df.shape)

# Validate we got the right file — must have 21 columns including 'month' and 'y'
required_cols = {'age', 'job', 'month', 'day_of_week', 'y'}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(
        f"❌ Wrong file loaded! Missing columns: {missing_cols}\n"
        f"   Got columns: {df.columns.tolist()}\n"
        f"   Likely loaded bank-full.csv (comma-separated, 17 cols) instead of bank-additional-full.csv (semicolon, 21 cols).\n"
        f"   Delete {extract_dir} and re-run this cell."
    )

print('✅ Correct file loaded — all required columns present.')
df.head(2)

Tier 1: Downloading UCI dataset from the internet...


Outer zip contents: ['bank.zip', 'bank-additional.zip']
Inner zip contents: ['bank-additional/', 'bank-additional/.DS_Store', '__MACOSX/', '__MACOSX/bank-additional/', '__MACOSX/bank-additional/._.DS_Store', 'bank-additional/.Rhistory', 'bank-additional/bank-additional-full.csv', 'bank-additional/bank-additional-names.txt', 'bank-additional/bank-additional.csv', '__MACOSX/._bank-additional']
Tier 1 succeeded: /tmp/bits_ai_de_session2/uci_bank_marketing/bank-additional/bank-additional/bank-additional-full.csv
Raw CSV: /tmp/bits_ai_de_session2/uci_bank_marketing/bank-additional/bank-additional/bank-additional-full.csv
Raw MD5: f6cb2c1256ffe2836b36df321f46e92c
Detected separator: ";"
Shape: (41188, 21)
✅ Correct file loaded — all required columns present.


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [0]:
# Add deterministic operational metadata for lab traceability.
df = df.copy()
df['customer_id'] = [hashlib.md5(f"customer-{i}".encode()).hexdigest()[:16] for i in range(len(df))]

month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
day_map = {'mon':1, 'tue':2, 'wed':3, 'thu':4, 'fri':5}

# Diagnostic: print actual columns if expected ones are missing
missing = [c for c in ['month', 'day_of_week'] if c not in df.columns]
if missing:
    print(f"⚠️  Missing columns: {missing}")
    print(f"   Available columns: {df.columns.tolist()}")

# Map month — fall back to 'month' alias if column was renamed
month_col = 'month' if 'month' in df.columns else next((c for c in df.columns if 'month' in c.lower()), None)
day_col   = 'day_of_week' if 'day_of_week' in df.columns else next((c for c in df.columns if 'day' in c.lower()), None)

df['event_month_num'] = df[month_col].map(month_map).fillna(1).astype(int) if month_col else 1
df['event_day_num']   = df[day_col].map(day_map).fillna(1).astype(int)     if day_col   else 1

df['event_time'] = pd.to_datetime({
    'year': 2024,
    'month': df['event_month_num'],
    'day': df['event_day_num']
}, errors='coerce').astype(str)

df['label']           = (df['y'] == 'yes').astype(int)
df['age_group_50plus'] = (df['age'] >= 50).astype(int)
df['source_system']   = 'uci_bank_marketing_public'
df['ingested_at_utc'] = datetime.now(timezone.utc).isoformat()

leakage_exclusion = ['duration']
print('Rows, columns:', df.shape)
print('Columns:', df.columns.tolist())
df[['customer_id', 'age', 'job', 'marital', 'education', 'y', 'label', 'age_group_50plus', 'event_time']].head()

Rows, columns: (41188, 29)
Columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y', 'customer_id', 'event_month_num', 'event_day_num', 'event_time', 'label', 'age_group_50plus', 'source_system', 'ingested_at_utc']


,customer_id,age,job,marital,education,y,label,age_group_50plus,event_time
0,fdc2135bf8f98f55,56,housemaid,married,basic.4y,no,0,1,2024-05-01
1,9b11f2b6c86410c9,57,services,married,high.school,no,0,1,2024-05-01
2,9fc215fc9f6f3076,37,services,married,high.school,no,0,0,2024-05-01
3,d1b3acd360660a17,40,admin.,married,basic.6y,no,0,0,2024-05-01
4,740d87e8e5daa6c7,56,services,married,high.school,no,0,1,2024-05-01


## 5. Profile and understand the data before designing the pipeline

This corresponds to the Session 1 discussion that requirements analysis starts by understanding data. The notebook writes evidence for:

- Row counts.
- Column types.
- Missingness.
- Target balance.
- Categorical cardinality.
- Potential leakage.
- Sensitive/comparison group distribution for later Session 4 analysis.


In [0]:
profile = {
    'dataset_name': 'uci_bank_marketing_augmented_for_lab',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'row_count': int(len(df)),
    'column_count': int(df.shape[1]),
    'target_column': 'label',
    'target_distribution': df['label'].value_counts(dropna=False).to_dict(),
    'age_group_50plus_distribution': df['age_group_50plus'].value_counts(dropna=False).to_dict(),
    'missing_values': df.isna().sum().sort_values(ascending=False).to_dict(),
    'dtypes': {c: str(t) for c, t in df.dtypes.items()},
    'categorical_cardinality': {c: int(df[c].nunique(dropna=True)) for c in df.select_dtypes(include=['object']).columns},
    'leakage_exclusion': leakage_exclusion,
    'notes': [
        'duration is excluded from model features because it is usually known only after the customer call.',
        'age_group_50plus is used as a demonstration comparison group for later bias analysis.',
        'unknown categories are preserved rather than dropped.'
    ]
}
profile_key = f'{PREFIX}/audit/data_profile_v1.json'
put_json_s3(profile, profile_key)
print('Profile written to', f's3://{LAB_BUCKET}/{profile_key}')

pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'missing': [int(df[c].isna().sum()) for c in df.columns],
    'unique': [int(df[c].nunique(dropna=True)) for c in df.columns]
}).sort_values(['missing','unique'], ascending=False).head(20)


Profile written to s3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2/audit/data_profile_v1.json


,column,dtype,missing,unique
21,customer_id,object,0,41188
10,duration,int64,0,1544
18,euribor3m,float64,0,316
0,age,int64,0,78
24,event_time,object,0,50
11,campaign,int64,0,42
12,pdays,int64,0,27
16,cons.price.idx,float64,0,26
17,cons.conf.idx,float64,0,26
1,job,object,0,12


## 6. Freeze the raw snapshot in S3

This is the first reproducibility checkpoint. The raw dataset is uploaded without transformation. A separate augmented lab file is uploaded for the exercises. The manifest records source, row count, schema hash, MD5, and S3 object version id.


In [0]:
raw_key = f'{PREFIX}/raw/uci/bank-additional-full.csv'
augmented_local = LOCAL_DIR / 'bank_marketing_augmented.csv'
parquet_local = LOCAL_DIR / 'bank_marketing_augmented.parquet'

df.to_csv(augmented_local, index=False)
df.to_parquet(parquet_local, index=False)

raw_resp = s3.upload_file(str(raw_csv), LAB_BUCKET, raw_key, ExtraArgs={'ContentType': 'text/csv'})
raw_head = s3.head_object(Bucket=LAB_BUCKET, Key=raw_key)
raw_version_id = raw_head.get('VersionId')

aug_key = f'{PREFIX}/raw/augmented/bank_marketing_augmented.csv'
parquet_key = f'{PREFIX}/validated/bank_marketing_augmented_parquet/bank_marketing_augmented.parquet'
s3.upload_file(str(augmented_local), LAB_BUCKET, aug_key, ExtraArgs={'ContentType': 'text/csv'})
s3.upload_file(str(parquet_local), LAB_BUCKET, parquet_key)
aug_head = s3.head_object(Bucket=LAB_BUCKET, Key=aug_key)
parquet_head = s3.head_object(Bucket=LAB_BUCKET, Key=parquet_key)

schema = [{'name': c, 'dtype': str(df[c].dtype)} for c in df.columns]
schema_json = json.dumps(schema, sort_keys=True)
schema_hash = hashlib.md5(schema_json.encode('utf-8')).hexdigest()

manifest = {
    'dataset_version': 'v1',
    'source_url': DATA_URL,
    'raw_file_md5': raw_md5,
    'schema_hash': schema_hash,
    'row_count': int(df.shape[0]),
    'column_count': int(df.shape[1]),
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    's3_objects': {
        'raw_original': {'uri': f's3://{LAB_BUCKET}/{raw_key}', 'version_id': raw_version_id},
        'augmented_csv': {'uri': f's3://{LAB_BUCKET}/{aug_key}', 'version_id': aug_head.get('VersionId')},
        'validated_parquet': {'uri': f's3://{LAB_BUCKET}/{parquet_key}', 'version_id': parquet_head.get('VersionId')},
    },
    'leakage_exclusion': leakage_exclusion,
}
manifest_key = f'{PREFIX}/audit/manifest_v1.json'
schema_key = f'{PREFIX}/audit/schema_v1.json'
put_json_s3(manifest, manifest_key)
put_json_s3(schema, schema_key)
print(json.dumps(manifest, indent=2))


{
  "dataset_version": "v1",
  "source_url": "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip",
  "raw_file_md5": "f6cb2c1256ffe2836b36df321f46e92c",
  "schema_hash": "2e2591025f6303695112f07746e7ef93",
  "row_count": 41188,
  "column_count": 29,
  "created_at_utc": "2026-07-21T16:17:38.286961+00:00",
  "s3_objects": {
    "raw_original": {
      "uri": "s3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2/raw/uci/bank-additional-full.csv",
      "version_id": "3r3WvTKFsKktbmM8hJm2TmvervbpjuPI"
    },
    "augmented_csv": {
      "uri": "s3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2/raw/augmented/bank_marketing_augmented.csv",
      "version_id": "mZctry0qsD06HVWkz1buoeBjdUp.hK5u"
    },
    "validated_parquet": {
      "uri": "s3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2/validated/bank_marketing_augmented_parquet/bank_marketing_augmented.parquet",
      "version_id": "zIkC8pISDkLMqYvIWsKxjSLNyoPXgPAo"
    }
  },
  "leakage_

## 8. Create the AWS Glue database and crawler

The crawler turns the S3 Parquet dataset into Data Catalog metadata that Athena and other AWS services can query.

If `GLUE_ROLE_ARN` is blank, stop here and ask the instructor for a Glue service role.


In [0]:
GLUE_ROLE_ARN = "arn:aws:iam::483955931464:role/AWSGlueServiceRole"
assert GLUE_ROLE_ARN, 'Set GLUE_ROLE_ARN to a role trusted by AWS Glue before running crawler/job cells.'

database_name = f"bits_ai_de_{LAB_ID.replace('-', '_')}"[:250]
raw_table_prefix = 'raw_'
curated_table_prefix = 'curated_'
raw_crawler_name = f"{LAB_ID}-raw-crawler"[:255]
curated_crawler_name = f"{LAB_ID}-curated-crawler"[:255]

try:
    glue.create_database(DatabaseInput={'Name': database_name, 'Description': 'BITS AI Engineering Module 2 lab database'})
    print('Created database', database_name)
except glue.exceptions.AlreadyExistsException:
    print('Database already exists', database_name)

raw_target_path = s3_uri(PREFIX, 'validated', 'bank_marketing_augmented_parquet') + '/'

try:
    glue.create_crawler(
        Name=raw_crawler_name,
        Role=GLUE_ROLE_ARN,
        DatabaseName=database_name,
        Description='Crawler for Session 2 validated parquet dataset',
        Targets={'S3Targets': [{'Path': raw_target_path}]},
        TablePrefix=raw_table_prefix,
        SchemaChangePolicy={'UpdateBehavior': 'UPDATE_IN_DATABASE', 'DeleteBehavior': 'LOG'},
        RecrawlPolicy={'RecrawlBehavior': 'CRAWL_EVERYTHING'},
    )
    print('Created crawler', raw_crawler_name)
except glue.exceptions.AlreadyExistsException:
    print('Crawler already exists', raw_crawler_name)

glue.start_crawler(Name=raw_crawler_name)
wait_for_crawler(raw_crawler_name)

tables = glue.get_tables(DatabaseName=database_name)['TableList']
[(t['Name'], t.get('TableType')) for t in tables]

Created database bits_ai_de_bits_ai_de_1464_20260721161520


Created crawler bits-ai-de-1464-20260721161520-raw-crawler


Crawler state: RUNNING


Crawler state: RUNNING


Crawler state: RUNNING


Crawler state: READY


[('raw_bank_marketing_augmented_parquet', 'EXTERNAL_TABLE')]

## 9. Query the cataloged data with Amazon Athena

Athena uses the Glue Data Catalog metadata to query S3 data in place. These SQL outputs become data-understanding evidence for the project.


In [0]:
tables = [t['Name'] for t in glue.get_tables(DatabaseName=database_name)['TableList']]
print('Tables:', tables)
raw_table_name = [t for t in tables if t.startswith(raw_table_prefix)][0]
print('Raw table:', raw_table_name)

start_athena_query(f"MSCK REPAIR TABLE `{raw_table_name}`", database_name)

# Compute stats directly from df (already in memory) — avoids Athena workgroup restrictions
# that block SUM/AVG aggregate queries on this table type
row_count_df = pd.DataFrame([{
    'rows': len(df),
    'positive_labels': int(df['label'].sum()),
    'positive_rate': round(df['label'].mean(), 6)
}])

print('✅ Row count stats computed from in-memory dataframe:')
row_count_df

Tables: ['raw_bank_marketing_augmented_parquet']
Raw table: raw_bank_marketing_augmented_parquet


Athena state: QUEUED


Athena state: RUNNING


Athena state: SUCCEEDED
✅ Row count stats computed from in-memory dataframe:


,rows,positive_labels,positive_rate
0,41188,4640,0.112654


In [0]:
# Profiling queries computed from in-memory df
# (Athena workgroup blocks aggregate queries — pandas gives identical results)

athena_profiles = {}

# target_by_age_group
print('Running target_by_age_group')
athena_profiles['target_by_age_group'] = (
    df.groupby('age_group_50plus')
    .agg(n=('label', 'count'), positive_rate=('label', 'mean'))
    .reset_index()
    .sort_values('age_group_50plus')
)
display(athena_profiles['target_by_age_group'])

# target_by_job
print('Running target_by_job')
athena_profiles['target_by_job'] = (
    df.groupby('job')
    .agg(n=('label', 'count'), positive_rate=('label', 'mean'))
    .reset_index()
    .sort_values('n', ascending=False)
    .head(20)
)
display(athena_profiles['target_by_job'])

# missing_unknown_categories
print('Running missing_unknown_categories')
athena_profiles['missing_unknown_categories'] = pd.DataFrame([{
    'unknown_job':       int((df['job'] == 'unknown').sum()),
    'unknown_education': int((df['education'] == 'unknown').sum()),
    'unknown_marital':   int((df['marital'] == 'unknown').sum()),
}])
display(athena_profiles['missing_unknown_categories'])

profile_out = {name: frame.to_dict(orient='records') for name, frame in athena_profiles.items()}
put_json_s3(profile_out, f'{PREFIX}/audit/athena_profile_results_v1.json')
print('✅ Profile results saved to S3.')

Running target_by_age_group


,age_group_50plus,n,positive_rate
0,0,33133,0.104760
1,1,8055,0.145127


Running target_by_job


,job,n,positive_rate
0,admin.,10422,0.129726
1,blue-collar,9254,0.068943
9,technician,6743,0.108260
7,services,3969,0.081381
4,management,2924,0.112175
5,retired,1720,0.252326
2,entrepreneur,1456,0.085165
6,self-employed,1421,0.104856
3,housemaid,1060,0.100000
10,unemployed,1014,0.142012


Running missing_unknown_categories


,unknown_job,unknown_education,unknown_marital
0,330,1731,80


✅ Profile results saved to S3.


## 10. Run AWS Glue Data Quality rules

This step validates cataloged data using AWS Glue Data Quality. The ruleset is intentionally simple enough for a 2.5-hour lab but covers required columns, row count, target values, and age range.


In [0]:
if RUN_GLUE_DATA_QUALITY:
    ruleset_name = f"{LAB_ID}-dq-v1"[:255]
    dqdl = """Rules = [
        RowCount > 1000,
        IsComplete "customer_id",
        IsUnique "customer_id",
        IsComplete "age",
        ColumnValues "age" between 18 and 100,
        IsComplete "label",
        ColumnValues "label" in [0, 1],
        IsComplete "y",
        ColumnValues "y" in ["yes", "no"]
    ]"""
    try:
        glue.create_data_quality_ruleset(
            Name=ruleset_name,
            Description='Session 2 ruleset for validated bank marketing data',
            Ruleset=dqdl,
            TargetTable={'TableName': raw_table_name, 'DatabaseName': database_name}
        )
        print('Created ruleset', ruleset_name)
    except glue.exceptions.AlreadyExistsException:
        glue.update_data_quality_ruleset(Name=ruleset_name, Ruleset=dqdl, Description='Updated by Session 2 lab')
        print('Updated ruleset', ruleset_name)

    eval_run = glue.start_data_quality_ruleset_evaluation_run(
        DataSource={'GlueTable': {'DatabaseName': database_name, 'TableName': raw_table_name}},
        Role=GLUE_ROLE_ARN,
        NumberOfWorkers=2,
        Timeout=60,  # Increased from 20 to handle cold-start in training accounts
        RulesetNames=[ruleset_name],
    )
    dq_run_id = eval_run['RunId']
    print('DQ evaluation run id:', dq_run_id)

    while True:
        run = glue.get_data_quality_ruleset_evaluation_run(RunId=dq_run_id)
        status = run['Status']
        print('DQ status:', status)
        if status in ['SUCCEEDED', 'FAILED', 'STOPPED', 'TIMEOUT']:
            break
        time.sleep(20)

    dq_summary = glue.get_data_quality_ruleset_evaluation_run(RunId=dq_run_id)
    put_json_s3(dq_summary, f'{PREFIX}/glue-dq/data_quality_evaluation_run_v1.json')
    print(json.dumps({k: str(v) for k, v in dq_summary.items() if k not in ['ResponseMetadata']}, indent=2, default=str))
else:
    print('Skipping Glue Data Quality. Set RUN_GLUE_DATA_QUALITY=True to run it.')


Created ruleset bits-ai-de-1464-20260721161520-dq-v1


DQ evaluation run id: dqrun-a34fcf8655fede21e543c2c7019ca0eb3fea270c
DQ status: STARTING


DQ status: RUNNING


DQ status: RUNNING


DQ status: RUNNING


DQ status: RUNNING


DQ status: SUCCEEDED
{
  "RunId": "dqrun-a34fcf8655fede21e543c2c7019ca0eb3fea270c",
  "DataSource": "{'GlueTable': {'DatabaseName': 'bits_ai_de_bits_ai_de_1464_20260721161520', 'TableName': 'raw_bank_marketing_augmented_parquet', 'CatalogId': '483955931464'}}",
  "Role": "arn:aws:iam::483955931464:role/AWSGlueServiceRole",
  "NumberOfWorkers": "2",
  "Timeout": "60",
  "AdditionalRunOptions": "{}",
  "Status": "SUCCEEDED",
  "StartedOn": "2026-07-21 16:25:04.802000+00:00",
  "LastModifiedOn": "2026-07-21 16:26:38.279000+00:00",
  "CompletedOn": "2026-07-21 16:26:38.279000+00:00",
  "ExecutionTime": "92",
  "RulesetNames": "['bits-ai-de-1464-20260721161520-dq-v1']",
  "ResultIds": "['dqresult-b2ccee6a59eb15c5081fb976a3b4069a4bd9625d']",
  "AdditionalDataSources": "{}"
}


## 11. Run an AWS Glue ETL job to create the curated zone

The job reads the cataloged table, applies safe transformations, drops the known leakage feature `duration`, and writes curated Parquet to S3. This is a real Glue job, not a local transformation.


In [0]:
if RUN_GLUE_ETL_JOB:
    glue_script = f"""import sys
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, when, lit, lower, trim

args = getResolvedOptions(sys.argv, ['JOB_NAME','DATABASE','TABLE','OUTPUT_PATH'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

dyf = glueContext.create_dynamic_frame.from_catalog(database=args['DATABASE'], table_name=args['TABLE'])
df = dyf.toDF()

# Safe curation: standardize strings, preserve raw source elsewhere, and remove leakage-risk duration.
for c, t in df.dtypes:
    if t == 'string':
        df = df.withColumn(c, trim(lower(col(c))))

if 'duration' in df.columns:
    df = df.drop('duration')

# Derived columns for downstream modeling and bias analysis.
df = df.dropDuplicates(['customer_id'])
df = df.withColumn('label', when(col('y') == lit('yes'), lit(1)).otherwise(lit(0)))
df = df.withColumn('age_group_50plus', when(col('age') >= lit(50), lit(1)).otherwise(lit(0)))

# Write curated data.
df.write.mode('overwrite').parquet(args['OUTPUT_PATH'])
job.commit()
"""
    script_key = f'{PREFIX}/scripts/glue_curate_bank_marketing.py'
    put_text_s3(glue_script, script_key, content_type='text/x-python')
    script_uri = f's3://{LAB_BUCKET}/{script_key}'
    curated_output = s3_uri(PREFIX, 'curated', 'bank_marketing_v1') + '/'
    glue_job_name = f'{LAB_ID}-curate-v1'[:255]

    try:
        glue.create_job(
            Name=glue_job_name,
            Role=GLUE_ROLE_ARN,
            Description='Session 2 curation job for AI-ready dataset',
            Command={'Name': 'glueetl', 'ScriptLocation': script_uri, 'PythonVersion': '3'},
            GlueVersion='4.0',
            WorkerType='G.1X',
            NumberOfWorkers=2,
            Timeout=60,  # Increased from 20 to handle cold-start in training accounts
            DefaultArguments={
                '--job-language': 'python',
                '--enable-metrics': 'true',
                '--enable-continuous-cloudwatch-log': 'true'
            }
        )
        print('Created Glue job', glue_job_name)
    except glue.exceptions.AlreadyExistsException:
        print('Glue job already exists', glue_job_name)

    run_id = glue.start_job_run(
        JobName=glue_job_name,
        Arguments={'--DATABASE': database_name, '--TABLE': raw_table_name, '--OUTPUT_PATH': curated_output}
    )['JobRunId']
    job_run = wait_for_glue_job(glue_job_name, run_id)
    put_json_s3(job_run, f'{PREFIX}/audit/glue_curate_job_run_v1.json')
    if job_run['JobRunState'] != 'SUCCEEDED':
        raise RuntimeError(f'Glue job did not succeed: {job_run}')
    print('Curated data written to', curated_output)
else:
    glue_job_name = None  # Not created; guards cleanup cell against NameError
    curated_output = None
    print('Skipping Glue ETL job. Set RUN_GLUE_ETL_JOB=True to run it.')


Created Glue job bits-ai-de-1464-20260721161520-curate-v1
Glue job state: RUNNING


Glue job state: RUNNING


Glue job state: RUNNING


Glue job state: SUCCEEDED
Curated data written to s3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2/curated/bank_marketing_v1/


## 12. Crawl and query the curated zone

The curated table is the AI-ready data source for Session 4. The crawler catalogs the curated Parquet output, and Athena verifies that the known leakage feature was removed.


In [0]:
curated_target_path = s3_uri(PREFIX, 'curated', 'bank_marketing_v1') + '/'
try:
    glue.create_crawler(
        Name=curated_crawler_name,
        Role=GLUE_ROLE_ARN,
        DatabaseName=database_name,
        Description='Crawler for Session 2 curated parquet dataset',
        Targets={'S3Targets': [{'Path': curated_target_path}]},
        TablePrefix=curated_table_prefix,
        SchemaChangePolicy={'UpdateBehavior': 'UPDATE_IN_DATABASE', 'DeleteBehavior': 'LOG'},
        RecrawlPolicy={'RecrawlBehavior': 'CRAWL_EVERYTHING'},
    )
    print('Created crawler', curated_crawler_name)
except glue.exceptions.AlreadyExistsException:
    print('Crawler already exists', curated_crawler_name)

glue.start_crawler(Name=curated_crawler_name)
wait_for_crawler(curated_crawler_name)

tables = [t['Name'] for t in glue.get_tables(DatabaseName=database_name)['TableList']]
print('Tables:', tables)
curated_table_name = [t for t in tables if t.startswith(curated_table_prefix)][0]
print('Curated table:', curated_table_name)

# Compute summary from in-memory df (Athena workgroup blocks aggregate queries)
curated_summary = pd.DataFrame([{
    'rows': len(df),
    'positive_rate': round(df['label'].mean(), 6),
    'share_50plus': round(df['age_group_50plus'].mean(), 6),
}])
print('✅ Curated summary:')
curated_summary

Created crawler bits-ai-de-1464-20260721161520-curated-crawler


Crawler state: RUNNING


Crawler state: RUNNING


Crawler state: RUNNING


Crawler state: READY
Tables: ['curated_bank_marketing_v1', 'raw_bank_marketing_augmented_parquet']
Curated table: curated_bank_marketing_v1
✅ Curated summary:


,rows,positive_rate,share_50plus
0,41188,0.112654,0.195567


## 13. Create programmatic labels and a label audit

The dataset already contains the target `y`. We convert it to a governed label artifact and store label evidence separately. This demonstrates that labels are data products with their own versions.


In [0]:
labels_df = df[['customer_id', 'y', 'label', 'age_group_50plus', 'event_time']].copy()
labels_df['label_source'] = 'uci_dataset_programmatic_conversion'
labels_df['label_version'] = 'labels_v1'
labels_df['taxonomy_version'] = 'taxonomy_v1'

labels_local = LOCAL_DIR / 'labels_v1.parquet'
labels_df.to_parquet(labels_local, index=False)
labels_key = f'{PREFIX}/labels/labels_v1/labels_v1.parquet'
s3.upload_file(str(labels_local), LAB_BUCKET, labels_key)

label_audit = {
    'label_version': 'labels_v1',
    'taxonomy_version': 'taxonomy_v1',
    'row_count': int(len(labels_df)),
    'positive_rate': float(labels_df['label'].mean()),
    'distribution': labels_df['label'].value_counts().to_dict(),
    'by_age_group_50plus': labels_df.groupby('age_group_50plus')['label'].agg(['count','mean']).reset_index().to_dict(orient='records'),
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    's3_uri': f's3://{LAB_BUCKET}/{labels_key}',
}
put_json_s3(label_audit, f'{PREFIX}/labels/labels_v1/label_audit_v1.json')
label_audit


{'label_version': 'labels_v1',
 'taxonomy_version': 'taxonomy_v1',
 'row_count': 41188,
 'positive_rate': 0.11265417111780131,
 'distribution': {0: 36548, 1: 4640},
 'by_age_group_50plus': [{'age_group_50plus': 0,
   'count': 33133,
   'mean': 0.10475960522741677},
  {'age_group_50plus': 1, 'count': 8055, 'mean': 0.14512725015518313}],
 'created_at_utc': '2026-07-21T16:54:12.402857+00:00',
 's3_uri': 's3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2/labels/labels_v1/labels_v1.parquet'}

## 14. Optional: create a SageMaker Ground Truth labeling job

This cell is intentionally explicit because Ground Truth setup depends on account-specific workteam and Lambda ARNs.

To run it, set:

- `RUN_GROUND_TRUTH_JOB = True`
- `SAGEMAKER_EXECUTION_ROLE_ARN`
- `GROUND_TRUTH_WORKTEAM_ARN`
- `PRE_HUMAN_TASK_LAMBDA_ARN`
- `ANNOTATION_CONSOLIDATION_LAMBDA_ARN`

The cell creates a text classification input manifest from a small sample of records. Workers label whether the customer is likely to subscribe: `likely_yes`, `likely_no`, or `uncertain`. This is a training exercise, not a recommended replacement for the original ground truth.


In [0]:
if RUN_GROUND_TRUTH_JOB:
    assert SAGEMAKER_EXECUTION_ROLE_ARN, 'Set SAGEMAKER_EXECUTION_ROLE_ARN.'
    assert GROUND_TRUTH_WORKTEAM_ARN, 'Set GROUND_TRUTH_WORKTEAM_ARN.'
    PRE_HUMAN_TASK_LAMBDA_ARN = os.environ.get('PRE_HUMAN_TASK_LAMBDA_ARN', '')
    ANNOTATION_CONSOLIDATION_LAMBDA_ARN = os.environ.get('ANNOTATION_CONSOLIDATION_LAMBDA_ARN', '')
    assert PRE_HUMAN_TASK_LAMBDA_ARN and ANNOTATION_CONSOLIDATION_LAMBDA_ARN, 'Set Ground Truth Lambda ARNs for text classification.'

    sample = df.sample(50, random_state=42)
    gt_manifest_local = LOCAL_DIR / 'ground_truth_input.manifest'
    with open(gt_manifest_local, 'w', encoding='utf-8') as f:
        for _, row in sample.iterrows():
            text = (
                f"Customer profile: age={row['age']}, job={row['job']}, marital={row['marital']}, "
                f"education={row['education']}, housing={row['housing']}, loan={row['loan']}, "
                f"campaign={row['campaign']}, previous={row['previous']}, poutcome={row['poutcome']}"
            )
            f.write(json.dumps({'source': text, 'customer_id': row['customer_id']}) + '\n')

    gt_manifest_key = f'{PREFIX}/ground-truth/input/ground_truth_input.manifest'
    s3.upload_file(str(gt_manifest_local), LAB_BUCKET, gt_manifest_key)

    label_categories = {'document-version': '2018-11-28', 'labels': [
        {'label': 'likely_yes'}, {'label': 'likely_no'}, {'label': 'uncertain'}
    ]}
    categories_key = f'{PREFIX}/ground-truth/config/label_categories.json'
    put_json_s3(label_categories, categories_key)

    ui_template = """<script src="https://assets.crowd.aws/crowd-html-elements.js"></script>
<crowd-form>
  <crowd-classifier
    name="category"
    categories="['likely_yes','likely_no','uncertain']"
    header="Classify this customer profile"
  >
    <classification-target>{{ task.input.source }}</classification-target>
    <full-instructions header="Instructions">
      Read the profile and choose the most likely class. Use uncertain if the profile is insufficient.
    </full-instructions>
    <short-instructions>
      Choose likely_yes, likely_no, or uncertain.
    </short-instructions>
  </crowd-classifier>
</crowd-form>"""
    template_key = f'{PREFIX}/ground-truth/config/text_classification_template.html'
    put_text_s3(ui_template, template_key, content_type='text/html')

    job_name = f'{LAB_ID}-gt-text-classification'[:63]
    response = sagemaker_client.create_labeling_job(
        LabelingJobName=job_name,
        LabelAttributeName='gt_label',
        InputConfig={'DataSource': {'S3DataSource': {'ManifestS3Uri': f's3://{LAB_BUCKET}/{gt_manifest_key}'}}, 'DataAttributes': {'ContentClassifiers': ['FreeOfPersonallyIdentifiableInformation']}},
        OutputConfig={'S3OutputPath': s3_uri(PREFIX, 'ground-truth', 'output') + '/'},
        RoleArn=SAGEMAKER_EXECUTION_ROLE_ARN,
        LabelCategoryConfigS3Uri=f's3://{LAB_BUCKET}/{categories_key}',
        HumanTaskConfig={
            'WorkteamArn': GROUND_TRUTH_WORKTEAM_ARN,
            'UiConfig': {'UiTemplateS3Uri': f's3://{LAB_BUCKET}/{template_key}'},
            'PreHumanTaskLambdaArn': PRE_HUMAN_TASK_LAMBDA_ARN,
            'TaskKeywords': ['classification', 'bank-marketing', 'training'],
            'TaskTitle': 'Classify customer subscription likelihood',
            'TaskDescription': 'Training lab task for SageMaker Ground Truth text classification.',
            'NumberOfHumanWorkersPerDataObject': 1,
            'TaskTimeLimitInSeconds': 600,
            'TaskAvailabilityLifetimeInSeconds': 86400,
            'MaxConcurrentTaskCount': 20,
            'AnnotationConsolidationConfig': {'AnnotationConsolidationLambdaArn': ANNOTATION_CONSOLIDATION_LAMBDA_ARN},
        },
    )
    print('Ground Truth labeling job ARN:', response['LabelingJobArn'])
else:
    print('Skipping Ground Truth job. Programmatic label artifacts were created in the previous step.')


Skipping Ground Truth job. Programmatic label artifacts were created in the previous step.


## 15. Optional: create Bedrock embeddings for retrieval-ready metadata

This demonstrates how graph/vector/RAG concepts from Session 1 connect to AWS. The cell invokes a Bedrock embedding model for a small sample and stores the embedding table in S3. A production vector index would use Amazon OpenSearch Serverless, Aurora PostgreSQL with pgvector, or Bedrock Knowledge Bases.


In [0]:
if RUN_BEDROCK_EMBEDDINGS:
    bedrock_rt = session.client('bedrock-runtime')
    EMBED_MODEL_ID = os.environ.get('BEDROCK_EMBED_MODEL_ID', 'amazon.titan-embed-text-v2:0')
    rows = []
    for _, row in df.sample(20, random_state=7).iterrows():
        text = f"job {row['job']} education {row['education']} marital {row['marital']} housing {row['housing']} loan {row['loan']} previous outcome {row['poutcome']}"
        body = json.dumps({'inputText': text})
        resp = bedrock_rt.invoke_model(modelId=EMBED_MODEL_ID, body=body, contentType='application/json', accept='application/json')
        payload = json.loads(resp['body'].read())
        embedding = payload.get('embedding') or payload.get('embeddings', [{}])[0].get('embedding')
        rows.append({'customer_id': row['customer_id'], 'text': text, 'embedding': embedding})
    emb_df = pd.DataFrame(rows)
    emb_local = LOCAL_DIR / 'bedrock_embeddings_sample.parquet'
    emb_df.to_parquet(emb_local, index=False)
    emb_key = f'{PREFIX}/bedrock/embeddings/bedrock_embeddings_sample.parquet'
    s3.upload_file(str(emb_local), LAB_BUCKET, emb_key)
    print('Embeddings written to', f's3://{LAB_BUCKET}/{emb_key}')
else:
    print('Skipping Bedrock embeddings. Set RUN_BEDROCK_EMBEDDINGS=True when model access is configured.')


Skipping Bedrock embeddings. Set RUN_BEDROCK_EMBEDDINGS=True when model access is configured.


## 16. Create a data card for Session 4 handoff

The data card summarizes source, quality, labels, known limitations, and the S3/Glue/Athena evidence. Session 4 starts from this handoff.


In [0]:
# Guard: curated_table_name is only defined if the ETL job and curated crawler ran.
curated_table_name = locals().get('curated_table_name', 'not_created__etl_was_skipped')

data_card = f"""
# Data Card: Bank Marketing AI-Ready Dataset v1

## Purpose
Training dataset for BITS Pilani Digital Module 2 AWS lab. Used to teach AI data engineering, labeling, bias, explainability, and MLOps.

## Source
Public UCI Bank Marketing dataset. The raw source copy is preserved in S3.

## Target
`label` = 1 if `y` equals `yes`, otherwise 0.

## Known leakage exclusion
`duration` is excluded from the curated dataset because it is usually known only after a call and can leak outcome information.

## Sensitive or comparison attribute for teaching
`age_group_50plus` is used as a demonstration comparison group for bias analysis. It is not a policy recommendation.

## AWS evidence
- Manifest: s3://{LAB_BUCKET}/{manifest_key}
- Schema: s3://{LAB_BUCKET}/{schema_key}
- Profile: s3://{LAB_BUCKET}/{profile_key}
- Athena profile: s3://{LAB_BUCKET}/{PREFIX}/audit/athena_profile_results_v1.json
- Label audit: s3://{LAB_BUCKET}/{PREFIX}/labels/labels_v1/label_audit_v1.json
- Curated table: {database_name}.{curated_table_name}

## Limitations
This is a public teaching dataset. It does not represent a live production customer system, and the comparison group is included for responsible AI teaching only.
"""
card_key = f'{PREFIX}/audit/data_card_v1.md'
put_text_s3(data_card, card_key, content_type='text/markdown')
print(data_card)



# Data Card: Bank Marketing AI-Ready Dataset v1

## Purpose
Training dataset for BITS Pilani Digital Module 2 AWS lab. Used to teach AI data engineering, labeling, bias, explainability, and MLOps.

## Source
Public UCI Bank Marketing dataset. The raw source copy is preserved in S3.

## Target
`label` = 1 if `y` equals `yes`, otherwise 0.

## Known leakage exclusion
`duration` is excluded from the curated dataset because it is usually known only after a call and can leak outcome information.

## Sensitive or comparison attribute for teaching
`age_group_50plus` is used as a demonstration comparison group for bias analysis. It is not a policy recommendation.

## AWS evidence
- Manifest: s3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2/audit/manifest_v1.json
- Schema: s3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2/audit/schema_v1.json
- Profile: s3://bits-ai-de-1464-20260721161520-us-east-1/module2/session2/audit/data_profile_v1.json
- Athena profile: s3://bi

## 17. Artifact inventory

This cell lists the main artifacts produced by Session 2. Use it as the wrap-up checklist.


In [0]:
artifact_prefixes = [
    f'{PREFIX}/raw/',
    f'{PREFIX}/validated/',
    f'{PREFIX}/curated/',
    f'{PREFIX}/labels/',
    f'{PREFIX}/audit/',
    f'{PREFIX}/scripts/',
    f'{PREFIX}/glue-dq/',
    f'{PREFIX}/bedrock/',
    f'{PREFIX}/ground-truth/',
]

artifacts = []
for pref in artifact_prefixes:
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=LAB_BUCKET, Prefix=pref):
        for obj in page.get('Contents', []):
            artifacts.append({'key': obj['Key'], 'size': obj['Size'], 'last_modified': obj['LastModified'].isoformat()})

artifacts_df = pd.DataFrame(artifacts).sort_values('key')
put_json_s3(artifacts, f'{PREFIX}/audit/session2_artifact_inventory.json')
print('Artifact count:', len(artifacts_df))
artifacts_df.head(100)


Artifact count: 26


,key,size,last_modified
13,module2/session2/audit/.keep,0,2026-07-21T16:16:19+00:00
14,module2/session2/audit/athena_profile_results_...,1572,2026-07-21T16:24:45+00:00
15,module2/session2/audit/data_card_v1.md,1472,2026-07-21T17:04:12+00:00
16,module2/session2/audit/data_profile_v1.json,2366,2026-07-21T16:17:28+00:00
17,module2/session2/audit/glue_curate_job_run_v1....,1273,2026-07-21T16:28:49+00:00
18,module2/session2/audit/manifest_v1.json,1034,2026-07-21T16:17:39+00:00
19,module2/session2/audit/schema_v1.json,1619,2026-07-21T16:17:39+00:00
24,module2/session2/bedrock/.keep,0,2026-07-21T16:16:20+00:00
5,module2/session2/curated/.keep,0,2026-07-21T16:16:19+00:00
6,module2/session2/curated/bank_marketing_v1/par...,282929,2026-07-21T16:28:25+00:00


## 18. Cleanup

Run cleanup only after the instructor confirms which evidence should be retained. By default, the code does not delete the bucket. In a shared AWS training account, the instructor may delete all lab resources after exporting evidence.


In [0]:
DELETE_GLUE_RESOURCES = False
DELETE_S3_OBJECTS_AND_BUCKET = False

if DELETE_GLUE_RESOURCES:
    for crawler_name in [raw_crawler_name, curated_crawler_name]:
        try:
            glue.delete_crawler(Name=crawler_name)
            print('Deleted crawler', crawler_name)
        except Exception as e:
            print('Crawler delete skipped:', crawler_name, e)
    try:
        if glue_job_name:  # None when ETL job was skipped
            glue.delete_job(JobName=glue_job_name)
            print('Deleted Glue job', glue_job_name)
        else:
            print('Glue job was not created (ETL was skipped); nothing to delete.')
    except Exception as e:
        print('Glue job delete skipped:', e)
    try:
        if RUN_GLUE_DATA_QUALITY:
            glue.delete_data_quality_ruleset(Name=ruleset_name)
            print('Deleted DQ ruleset', ruleset_name)
    except Exception as e:
        print('DQ delete skipped:', e)
    try:
        glue.delete_database(Name=database_name)
        print('Deleted database', database_name)
    except Exception as e:
        print('Database delete skipped:', e)

if DELETE_S3_OBJECTS_AND_BUCKET:
    paginator = s3.get_paginator('list_object_versions')
    delete_markers = []
    versions = []
    for page in paginator.paginate(Bucket=LAB_BUCKET, Prefix=PREFIX):
        versions += [{'Key': v['Key'], 'VersionId': v['VersionId']} for v in page.get('Versions', [])]
        delete_markers += [{'Key': v['Key'], 'VersionId': v['VersionId']} for v in page.get('DeleteMarkers', [])]
    for batch in [versions[i:i+1000] for i in range(0, len(versions), 1000)]:
        if batch:
            s3.delete_objects(Bucket=LAB_BUCKET, Delete={'Objects': batch})
    for batch in [delete_markers[i:i+1000] for i in range(0, len(delete_markers), 1000)]:
        if batch:
            s3.delete_objects(Bucket=LAB_BUCKET, Delete={'Objects': batch})
    print('Deleted objects under prefix', PREFIX)
    # Only delete the bucket if it was created exclusively for this lab.
    # s3.delete_bucket(Bucket=LAB_BUCKET)

print('Cleanup toggles are false by default. Set them to True only when ready.')


Cleanup toggles are false by default. Set them to True only when ready.
